# Notebook 2: Social Homophily Computation

This notebook computes sociodemographic **similarity** (cosine similarity) and **social connectedness** (Facebook SCI) between evacuees' home and destination census block groups.

**Paper reference:** Figure 2 — Social homophily and destination choice.

**Inputs:** `evacuees_for_homophily_final.csv`, `ACS_Colorado_2019_Block_Groups.csv`

**Outputs:** Similarity and connectedness scores per evacuee


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
evacuees =pd.read_csv("evacuess_for_homophily_final.csv")

In [ ]:
evacuees = evacuees.reset_index()
evacuees.head()

In [ ]:
columns_drop = ['D1_scaled_sci','D1_total_housing_units', 'D1_gravity' ]
evacuees = evacuees.drop(columns=columns_drop)
evacuees.columns

In [ ]:
df = evacuees.groupby('motif_final')['index'].nunique()
df

In [ ]:
census_org = pd.read_csv("Shapefile_census/ACS_Colorado_2019_Block_Groups.csv")

In [ ]:
census = census_org[(census_org['median_household_income'])>0]

In [ ]:
census['white_population'] = (census['white_population']  /census['total_population'] )*100
census['black_population'] = (census['black_population']  /census['total_population'] )*100
census['asian_population'] = (census['asian_population']  /census['total_population'] )*100
census['all_others'] = ((census['total_population']-(census['white_population']+census['black_population']+ census['asian_population']))
                                                    /census['total_population'])*100
census['education_atleast_one_degree'] = (census['total_population_25_over']  /census['total_population'] )*100

In [ ]:
census = census.rename(columns = {'total_housing_units' : 'total_housing_units_r'})

In [ ]:
columns_keep = ['total_housing_units_r','white_population', 'black_population', 'asian_population', 'education_atleast_one_degree', 
                'unemployment_rate', 'geoid', 'all_others' ]
census = census[columns_keep]
census.head()

In [ ]:
census_D1 = census.rename(
    columns=lambda x: 'Or_' + x if x != 'geoid' else x
)
census_D1.head()

In [ ]:
census_D1['geoid'] = census_D1['geoid'].astype(str)
evacuees['pre_crisis_home_cbg'] = evacuees['pre_crisis_home_cbg'].astype(str)

In [ ]:
evacuees = evacuees.merge(census_D1, left_on = "pre_crisis_home_cbg", right_on = "geoid", how = "inner")

In [ ]:
census_D2 = census.rename(
    columns=lambda x: 'D1_' + x if x != 'geoid' else x
)
census_D2.head()

In [ ]:
census_D2['geoid'] = census_D2['geoid'].astype(str)
evacuees['crisis_home_cbg'] = evacuees['crisis_home_cbg'].astype(str)
evacuees = evacuees.merge(census_D2, left_on = "crisis_home_cbg", right_on = "geoid", how = "inner")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

In [ ]:
origin_columns = [
       'Or_population_density', 
       'Or_white_population', 
       #'Or_all_others',
       'Or_black_population',
       #'Or_asian_population', 
       'Or_education_atleast_one_degree',
       'Or_unemployment_rate', 
       'Or_median_household_income'	   
]

destination_columns = [
        'D1_population_density', 
        'D1_white_population',
        #'D1_all_others',
        'D1_black_population', 
        #'D1_asian_population',
       'D1_education_atleast_one_degree', 
       'D1_unemployment_rate',
       'D1_median_household_income'
]

evacuees = evacuees.dropna()
# Normalize the data for origin and destination separately
# scaler = MinMaxScaler()
# normalized_origin = scaler.fit_transform(evacuees[origin_columns])
# normalized_destination = scaler.fit_transform(evacuees[destination_columns])
scaler = StandardScaler()
normalized_origin = scaler.fit_transform(evacuees[origin_columns])
normalized_destination = scaler.fit_transform(evacuees[destination_columns])
# Compute cosine similarity for each row
homophily_scores = []
for i in range(len(evacuees)):
    origin_vector = normalized_origin[i].reshape(1, -1)
    destination_vector = normalized_destination[i].reshape(1, -1)
    similarity = cosine_similarity(origin_vector, destination_vector)[0][0]
    homophily_scores.append(similarity)

# Append the homophily scores to the original DataFrame
evacuees['homophily'] = homophily_scores
evacuees.head()

In [ ]:
evacuees['homophily'].mean()

In [ ]:
evacuees.to_csv("evacuees_for_regression_final.csv")

In [ ]:
evacuees['homophily'].hist()

In [ ]:
evacuees.columns

In [ ]:
evacuees[['Or_total_population',
       'Or_median_household_income', 'Or_unemployment',
       'Or_total_housing_units', 'Or_occupied_housing_units',
       'Or_vacant_housing_units', 'avg_entropy', 'avg_predictability',
       'avg_exploration', 'radius_of_gyration', 'Sl_Num',
       'Or_white_population', 'Or_black_population', 'Or_asian_population',
       'Or_education_atleast_one_degree', 'Or_unemployment_rate',
       'homophily']].corr()